# Cost Controls for Custom Agents: a per-project budget that blocks runaway spend

An agent stuck in a retry loop, or a prompt that drags a huge context into every turn, can spend a
month of model budget in an afternoon. **Cost Controls** give each Highflame project a daily and a
monthly budget in US dollars. Highflame prices every model call it is told about, adds it to the
project's spend, and blocks new prompts once the budget is used up.

This notebook is for a **custom agent**: one that calls its model directly and uses the Highflame
SDK for guardrails. The agent is registered in the Highflame **Registry** and signs every call with
its own API key, so its spend is the agent's. It tells Highflame what each model call used with one
SDK call, `usage.report()`, and its usual guard check refuses the next prompt once the budget is
used up.

1. Read the project's budget, without spending anything.
2. Make one guarded model call that reports its usage.
3. Keep going until the budget is used up, and see the next prompt refused before it reaches the model.
4. See where the money went, by agent and by model.

Agents whose model calls go through the Highflame AI Gateway need no code at all; they have their own
notebook: [`01-ai-gateway`](../01-ai-gateway/).

## How it works

1. The agent checks the prompt with `guard.evaluate_prompt()`, as it already does for guardrails.
2. It calls its model directly.
3. It reports the call's token counts with `usage.report()`. Highflame prices the call from one price
   table, adds it to the project's spend for the day and for the month (UTC), and returns the price.
4. On the **next** prompt, the guard check runs the Cost Controls policy, which compares the spend
   with the budget and refuses once it is used up, before any more money is spent.

The price of a call is known only after the model answers, so the call that crosses the budget
completes and the one after it is refused. If that overshoot matters, use the **Monthly Budget
Headroom** template, which blocks at 95% of the budget.

## Before you start: set it up once in Studio

Use a project that nothing else depends on, and only this notebook. When the budget is used up,
**every** prompt in the project is blocked until the next UTC day, or until you raise the budget.

1. Sign in at [studio.highflame.ai](https://studio.highflame.ai) and create a project, for example
   `cost-controls-custom-agents`.
2. **Custom Agents → Policies → Cost Controls → Set budget.** Set **Daily budget (USD)** to `0.05`, leave
   the monthly budget blank, and select **Deploy**. The budget belongs to the project, so the Custom
   Agents and AI Gateway pages show the same one.
3. In the same tab, deploy the **Daily Budget** template. The deploy dialog starts on **Monitor**,
   which only records; set **Enforcement Mode** to **Enforce**.
4. **Registry → Register Identity.** Keep **Agent Identity** on, name the agent (for example `Support Summary Agent`), and select **Register Identity**. Copy the API key it shows; Studio shows it only once. This is the agent's own key: every guard check and usage report it signs belongs to the agent.

With `gpt-4o` each call below costs about one cent, so a $0.05 budget runs out in a handful of
calls. That is the point: the notebook should hit the limit in front of you.

## Setup

Copy `.env.example` to `.env` beside this notebook.

| Variable | What it is |
| --- | --- |
| `HIGHFLAME_API_KEY` | **Required.** The agent's API key from step 4. Use the agent's key, not an AI Gateway key, so the spend is the agent's. The next cell stops if the key is not an agent's. |
| `PROVIDER_API_KEY` | **Required.** Your OpenAI API key. The agent calls OpenAI with it directly. `OPENAI_API_KEY` works as a fallback. |
| `MODEL` | An OpenAI model. Defaults to `gpt-4o`, which costs enough per call to reach a small budget quickly. |
| `DAILY_BUDGET_USD` | Optional. The notebook reads the budget amount from Highflame; this is used only if Highflame does not report it. Defaults to `0.05`. |

In [1]:
%pip install -q -r requirements.txt

from importlib.metadata import version

print("highflame", version("highflame"))

Note: you may need to restart the kernel to use updated packages.
highflame 0.10.0


In [2]:
import json
import os
import time
import urllib.parse
import urllib.request
import uuid
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from highflame import Highflame
from openai import OpenAI

load_dotenv()

HIGHFLAME_API_KEY = os.environ["HIGHFLAME_API_KEY"]
PROVIDER_API_KEY = os.environ.get("PROVIDER_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
if not PROVIDER_API_KEY:
    raise RuntimeError("Set PROVIDER_API_KEY (or OPENAI_API_KEY) in .env to your OpenAI API key.")
MODEL = os.environ.get("MODEL", "gpt-4o")
DAILY_BUDGET_USD = float(os.environ.get("DAILY_BUDGET_USD", "0.05"))

# The platform API and the identity service. Leave both unset for the hosted product.
API_URL = os.environ.get("HIGHFLAME_API_URL", "https://api.highflame.ai").rstrip("/")
AUTH_URL = os.environ.get("HIGHFLAME_AUTH_URL", "https://auth.highflame.ai").rstrip("/")
SDK_URLS = {"base_url": API_URL, "identity_base_url": AUTH_URL, "token_url": f"{AUTH_URL}/oauth2/token"}

RUN_ID = uuid.uuid4().hex[:6]

highflame =Highflame(api_key=HIGHFLAME_API_KEY, **SDK_URLS)
identity = highflame.guard.evaluate_prompt("who am I", mode="monitor").agent_identity
if identity is None or identity.identity_type != "agent":
    raise RuntimeError(
        "HIGHFLAME_API_KEY is not an agent's key. Use the key from Registry -> Register Identity"
        " (step 4 of 'Before you start'), not an AI Gateway key."
    )
AGENT_ID = identity.external_id
print("agent  :", identity.name, f"({AGENT_ID}, {identity.status})")
print("model  :", MODEL)

agent  : Support Summary Agent (cost-controls-support-agent, active)
model  : gpt-4o


### Today's spend so far

Highflame keeps one record per priced call and serves them as a chargeback report. Read today's
totals now, before this run spends anything, so section 4 can show what this run added.

In [3]:
form = urllib.parse.urlencode({"grant_type": "api_key", "api_key": HIGHFLAME_API_KEY}).encode()
with urllib.request.urlopen(
    urllib.request.Request(f"{AUTH_URL}/oauth2/token", data=form, method="POST",
                           headers={"Content-Type": "application/x-www-form-urlencoded"}),
    timeout=30,
) as response:
    read_token = json.loads(response.read())["access_token"]


def chargeback(group_by: str) -> dict:
    now = datetime.now(timezone.utc)
    query = urllib.parse.urlencode({
        "start": now.strftime("%Y-%m-%dT00:00:00Z"),
        "end": (now + timedelta(minutes=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by": group_by,
    })
    request = urllib.request.Request(
        f"{API_URL}/v1/obs/costs/chargeback?{query}", headers={"Authorization": f"Bearer {read_token}"}
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read())


BASELINE = chargeback("agent")
CALLS_MADE = 0  # priced calls this run makes; section 4 waits until the report has all of them
print(f"Today so far: ${BASELINE.get('total_cost_usd', 0):.4f} over {BASELINE.get('total_calls', 0)} calls")

Today so far: $0.0000 over 0 calls


## 1. Read the budget, without spending anything

`budget_state()` asks Highflame how much of the project's budget is used, the same way a policy
sees it. It runs a guard check in `monitor` mode with `debug=True`, which returns the values the
Cost Controls policies read:

| Value | Meaning |
| --- | --- |
| `budget_cost_pct_daily` | Share of today's budget used, 0 to 100 |
| `budget_exceeded_daily` | `True` once today's budget is used up |
| `agent_spend_cents_daily` | What the calling agent spent today, in US cents |

They appear only when they apply: the budget values only when the project has a daily budget, and
the agent value only when the caller is a registered agent.

The budget amount itself comes from Highflame too, through a `dryrun` guard check that returns the
configuration of every detector the active policies need. Edit the budget in Studio and the next
read shows the new amount.

In [4]:
from highflame import GuardRequest

BUDGET_KEYS = (
    "budget_cost_pct_daily",
    "budget_exceeded_daily",
    "budget_cost_pct_monthly",
    "budget_exceeded_monthly",
    "agent_spend_cents_daily",
    "agent_spend_cents_monthly",
)


def budget_limits(client: Highflame = highflame) -> dict:
    # A dry run executes nothing: it returns the detectors the active policies need, with this
    # project's configuration for each. The budget checker's configuration is the budget.
    plan = client.guard.evaluate(GuardRequest(
        content="budget check", content_type="prompt", action="process_prompt", optimize=True, dryrun=True,
    )).optimization
    for detector in (plan.model_dump() if plan else {}).get("required_detectors") or []:
        if detector.get("name") == "budget_checker":
            return detector.get("config") or {}
    return {}


def budget_state(client: Highflame = highflame) -> dict:
    # mode="monitor": a read, never a block. The check costs nothing; it carries no tokens.
    decision = client.guard.evaluate_prompt("budget check", mode="monitor", debug=True)
    seen = {**(decision.context or {}), **(decision.projected_context or {})}
    state = {key: seen[key] for key in BUDGET_KEYS if key in seen}
    state["daily_usd"] = budget_limits(client).get("daily_usd", DAILY_BUDGET_USD)
    return state


def meter(state: dict) -> str:
    pct = int(state.get("budget_cost_pct_daily", 0))
    bar = "#" * (pct // 5) + "." * (20 - pct // 5)
    budget = state.get("daily_usd") or DAILY_BUDGET_USD
    used = budget * pct / 100
    flag = "  USED UP" if state.get("budget_exceeded_daily") else ""
    return f"[{bar}] {pct:3d}% of ${budget:.2f} (~${used:.3f}){flag}"


state = budget_state()
if "budget_cost_pct_daily" not in state:
    raise RuntimeError(
        "This project has no daily budget. Do step 2 of 'Before you start' in Studio, then run this cell again."
    )
print("project today:", meter(state))
if state.get("budget_exceeded_daily"):
    print(
        "\nToday's budget is already used up, probably by an earlier run. Raise the budget in Studio"
        " (Cost Controls -> Edit budget), or run this again after 00:00 UTC."
    )

project today: [....................]   0% of $0.15 (~$0.000)


## 2. One guarded model call that reports its usage

Two SDK calls matter here:

- `guard.evaluate_prompt()` **before** the model call. This is where a used-up budget is refused.
- `usage.report()` **after** the model call, with the token counts from the provider's response.
  This is the only addition the budget needs. Highflame prices the call and returns the price.

Report each call once. OpenAI counts cached prompt tokens inside `prompt_tokens`, so the agent sends
them separately as `cache_read_tokens`, where they are priced at the cached rate.

Every prompt below starts with a unique line. That keeps the provider's prompt cache out of the
way, so each call costs about the same.

In [5]:
# A support backlog to summarise: about 3,000 tokens of input, so each gpt-4o call costs about a cent.
ISSUES = ["login fails", "invoice is wrong", "export times out", "app crashes on start", "password reset email never arrives"]
PRODUCTS = ["the mobile app", "the web portal", "the billing page", "the admin console"]
BACKLOG = "\n".join(
    f"Ticket {n}: a customer reports that {ISSUES[n % 5]} in {PRODUCTS[n % 4]} since the last release."
    for n in range(1, 151)
)
TASK = "Write a short paragraph on each of the five most common problems in these tickets, with how many tickets mention it."


def work_prompt(step: int) -> str:
    return f"Request {step} of run {RUN_ID}.\n\n{TASK}\n\n{BACKLOG}"


openai_direct = OpenAI(api_key=PROVIDER_API_KEY)  # straight to OpenAI, no gateway
SESSION = f"cost-demo-{RUN_ID}"


def custom_agent_call(step: int) -> dict:
    global CALLS_MADE
    prompt = work_prompt(step)

    # 1. Guard the prompt. A used-up budget is refused here, before any money is spent.
    decision = highflame.guard.evaluate_prompt(prompt, session_id=SESSION)
    if not decision.allowed:
        return {"allowed": False, "tokens": "-", "cost": "-", "text": decision.policy_reason or ""}

    # 2. Call the model directly.
    reply = openai_direct.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=400
    )
    usage = reply.usage
    cached = getattr(usage.prompt_tokens_details, "cached_tokens", 0) or 0

    # 3. Report what the call used. Highflame prices it and adds it to the project's spend.
    report = highflame.usage.report(
        model=MODEL,
        provider="openai",
        input_tokens=usage.prompt_tokens - cached,
        cache_read_tokens=cached,
        output_tokens=usage.completion_tokens,
        session_id=SESSION,
    )
    CALLS_MADE += report.recorded
    return {
        "allowed": True,
        "tokens": f"{usage.prompt_tokens}/{usage.completion_tokens}",
        "cost": f"${report.cost_micros / 1e6:.4f}" + ("" if report.priced else " (model not priced)"),
        "text": reply.choices[0].message.content or "",
    }


result = custom_agent_call(1)
print("allowed:", result["allowed"], " tokens in/out:", result["tokens"], " priced by Highflame:", result["cost"])
print("reply  :", result["text"][:200].replace("\n", " "), "...")
print("\nproject today:", meter(budget_state()))

allowed: True  tokens in/out: 3100/354  priced by Highflame: $0.0113
reply  : After analyzing the tickets, five common problems emerge from the reports, each presenting significant challenges since the last release.  1. **Incorrect Invoices**: A total of 36 tickets mention issu ...



project today: [#...................]   7% of $0.15 (~$0.011)


## 3. Keep going until the budget is used up

Each call adds to the project's spend. Once the bar reaches 100%, the next prompt is blocked.

In [6]:
refusal = None
previous = budget_state()
for step in range(2, 31):  # stops at the first refusal; the cap only bounds a misconfigured run
    result = custom_agent_call(step)
    time.sleep(2)  # let the usage report land before the next read
    state = budget_state()
    print(
        f"{step:>2}  {'allowed' if result['allowed'] else 'BLOCKED':<8}"
        f"  tokens {result['tokens']:<9}  {result['cost']:<24}  {meter(state)}"
    )
    if not result["allowed"]:
        refusal = result["text"]
        break
    if previous.get("budget_exceeded_daily"):
        print(
            "\nThe budget was already used up, but this call was allowed. Check that the Daily Budget"
            " template is deployed in Enforce mode, not Monitor."
        )
        break
    previous = state

print("\nWhat the agent got back:\n ", (refusal or "(not blocked)")[:300])

 2  allowed   tokens 3100/337   $0.0111                   [##..................]  14% of $0.15 (~$0.021)


 3  allowed   tokens 3100/394   $0.0117                   [####................]  22% of $0.15 (~$0.033)


 4  allowed   tokens 3100/326   $0.0110                   [######..............]  30% of $0.15 (~$0.045)


 5  allowed   tokens 3100/400   $0.0118                   [#######.............]  37% of $0.15 (~$0.056)


 6  allowed   tokens 3100/400   $0.0118                   [#########...........]  45% of $0.15 (~$0.068)


 7  allowed   tokens 3100/315   $0.0109                   [##########..........]  53% of $0.15 (~$0.079)


... (4 outputs left out)

12  allowed   tokens 3100/367   $0.0114                   [##################..]  90% of $0.15 (~$0.135)


13  allowed   tokens 3100/377   $0.0115                   [###################.]  98% of $0.15 (~$0.147)


14  allowed   tokens 3100/388   $0.0116                   [####################] 100% of $0.15 (~$0.150)  USED UP


15  BLOCKED   tokens -          -                         [####################] 100% of $0.15 (~$0.150)  USED UP

What the agent got back:
  Enterprise Policies Triggered: Daily Budget


The guard check refused the prompt **before** the model call, so the refused prompt cost nothing.
The agent sees an ordinary guard decision, which it already handles for every other guardrail.

## 4. Where the money went

Highflame keeps one record per priced call, and serves it as a chargeback report grouped by agent,
by user, or by model. Studio shows the same report under **Observatory → Chargeback**.

The report is built from a stream, so the newest calls can take a minute or two to appear. The cell
waits until every call this run made is in it. The report covers whole UTC days, so the rows below
are today's totals; the first line is this run alone.

In [7]:
for _ in range(24):  # up to about 4 minutes
    by_agent = chargeback("agent")
    if by_agent.get("total_calls", 0) >= BASELINE.get("total_calls", 0) + CALLS_MADE:
        break
    time.sleep(10)


def show(title: str, report: dict) -> None:
    print(title)
    for row in report.get("rows", []):
        if row["cost_usd"] > 0:  # calls at $0 (an unpriced model, for one) carry no cost to explain
            label = row["key"] or "(not a registered agent)"
            print(f"  {label:<28} ${row['cost_usd']:.4f}  {row['calls']:>3} calls  {row['share']:.0%}")


run_cost = by_agent.get("total_cost_usd", 0) - BASELINE.get("total_cost_usd", 0)
print(f"This run: ${run_cost:.4f} over {CALLS_MADE} calls")
print(f"Today, this project: ${by_agent.get('total_cost_usd', 0):.4f} over {by_agent.get('total_calls', 0)} calls\n")
show("By agent (today)", by_agent)
show("\nBy model (today)", chargeback("model"))

This run: $0.1590 over 14 calls
Today, this project: $0.1590 over 14 calls

By agent (today)
  cost-controls-support-agent  $0.1590   14 calls  100%



By model (today)
  openai/gpt-4o                $0.1590   14 calls  100%


## 5. Per-agent budgets

The project budget caps the total. You can also cap one agent: Highflame counts each registered
agent's spend, and the **Agent Monthly Budget** template blocks an agent once it has spent $200 in a
month. Attach it to the agent and edit the threshold for that agent. Its spend still counts against
the project's budget too.

The cell below reads what the agent spent today, as that template sees it.

In [8]:
cents = budget_state().get("agent_spend_cents_daily")
print(f"{identity.name} spent today: {cents if cents is not None else '-'} cents")

Support Summary Agent spent today: 15 cents


## Run it again

The agent stays in the Registry, so the next run uses the same key and the same agent.

The project stays blocked for the rest of the UTC day. To run the notebook again today, raise the
budget in Studio (**Cost Controls → Edit budget**). To turn budgets off, remove the budget: with no
budget, the Cost Controls templates never fire.

## What this showed

- A daily budget in US dollars stopped a custom agent that calls its model directly, once the budget
  was used up.
- The agent needed one `usage.report()` call after each model call. Highflame priced every call from
  one price table and returned the price.
- The guard check the agent already makes refused the next prompt before it reached the model.
- The chargeback report named the agent and the model behind every cent.

**Good to know**

- Spend is known only after a model answers, so the call that crosses the budget completes. The
  Monthly Budget Headroom template blocks at 95% to keep the overshoot small.
- A model that is not in Highflame's price table is counted at $0, and `usage.report()` returns
  `priced: false` for it.
- Days and months are UTC.
- A project has one budget. Calls through the AI Gateway in the same project count against it too.